In [22]:
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow.keras import Model, layers
from tensorflow.keras.preprocessing.sequence import pad_sequences

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

corpus_path = Path("Int.txt")
if not corpus_path.is_file():
    # Generate a sample input.txt automatically so the notebook runs immediately
    sample_words = [
        "hello", "world", "sequence", "learning", "tensorflow",
        "keras", "encoder", "decoder", "recurrent", "network"
    ]
    corpus_path.write_text("\n".join(sample_words), encoding="utf-8")
    print("Created a sample input.txt with default words.")

lines = [line.strip().lower() for line in corpus_path.read_text(encoding="utf-8").splitlines()]
character_sequences = [line for line in lines if len(line) >= 3]

# If only one line is provided, duplicate it so that validation split doesn't crash
if len(character_sequences) == 1:
    character_sequences = character_sequences * 2
elif len(character_sequences) < 1:
    raise ValueError("The corpus must contain at least one non-empty line of length >= 3.")

characters = sorted(set("".join(character_sequences)))
char_to_id = {character: index for index, character in enumerate(characters, start=1)}
id_to_char = {index: character for character, index in char_to_id.items()}

PAD_ID = 0
START_ID = len(char_to_id) + 1
END_ID = START_ID + 1
VOCAB_SIZE = END_ID + 1

max_input_length = max(map(len, character_sequences))
encoder_data = pad_sequences(
    [[char_to_id[character] for character in sequence] for sequence in character_sequences],
    maxlen=max_input_length,
    padding="post",
    dtype="int32",
)

target_sequences = [
    [START_ID]
    + [char_to_id[character] for character in sequence[::-1]]
    + [END_ID]
    for sequence in character_sequences
]
decoder_length = max_input_length + 1
decoder_input_data = pad_sequences(
    [sequence[:-1] for sequence in target_sequences],
    maxlen=decoder_length,
    padding="post",
    dtype="int32",
)
decoder_target_data = pad_sequences(
    [sequence[1:] for sequence in target_sequences],
    maxlen=decoder_length,
    padding="post",
    dtype="int32",
)
target_weights = (decoder_target_data != PAD_ID).astype("float32")

rng = np.random.default_rng(SEED)
indices = rng.permutation(len(encoder_data))
validation_size = max(1, int(0.2 * len(indices))) if len(indices) > 1 else 0
validation_indices = indices[:validation_size] if validation_size > 0 else indices
training_indices = indices[validation_size:] if validation_size > 0 else indices

print(f"Examples: {len(character_sequences)}")
print(f"Character vocabulary (including reserved tokens): {VOCAB_SIZE}")
print(f"Maximum input length: {max_input_length}")

Examples: 5
Character vocabulary (including reserved tokens): 13
Maximum input length: 7


In [23]:
embedding_dim = 32
latent_dim = 128

encoder_tokens = layers.Input(shape=(max_input_length,), name="encoder_tokens")
token_embedding = layers.Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=embedding_dim,
    mask_zero=True,
    name="token_embedding",
)
encoder_lstm = layers.LSTM(latent_dim, return_state=True, name="encoder_lstm")
encoder_embedded = token_embedding(encoder_tokens)
_, encoder_state_h, encoder_state_c = encoder_lstm(encoder_embedded)
encoder_states = [encoder_state_h, encoder_state_c]

decoder_tokens = layers.Input(shape=(decoder_length,), name="decoder_tokens")
decoder_lstm = layers.LSTM(latent_dim, return_sequences=True, return_state=True, name="decoder_lstm")
decoder_embedded = token_embedding(decoder_tokens)
decoder_outputs, _, _ = decoder_lstm(decoder_embedded, initial_state=encoder_states)
token_classifier = layers.Dense(VOCAB_SIZE, activation="softmax", name="token_classifier")
decoder_probabilities = token_classifier(decoder_outputs)

model = Model([encoder_tokens, decoder_tokens], decoder_probabilities, name="lstm_seq2seq")
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["sparse_categorical_accuracy"],
)
model.summary()

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True,
)
history = model.fit(
    [encoder_data[training_indices], decoder_input_data[training_indices]],
    decoder_target_data[training_indices],
    sample_weight=target_weights[training_indices],
    validation_data=(
        [encoder_data[validation_indices], decoder_input_data[validation_indices]],
        decoder_target_data[validation_indices],
        target_weights[validation_indices],
    ),
    epochs=100,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1,
)
print(f"Best validation loss: {min(history.history['val_loss']):.4f}")

Model: "lstm_seq2seq"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ decoder_tokens      │ (None, 8)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_tokens      │ (None, 7)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ token_embedding     │ (None, 8, 32)     │        416 │ encoder_tokens[0… │
│ (Embedding)         │                   │            │ decoder_tokens[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_9         │ (None, 7)         │          0 │ encoder_tokens[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 128),     │     82,432 │ token_embedding[… │
│                     │ (None, 128),      │            │ not_equal_9[0][0] │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 8, 128),  │     82,432 │ token_embedding[… │
│                     │ (None, 128),      │            │ encoder_lstm[0][… │
│                     │ (None, 128)]      │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ token_classifier    │ (None, 8, 13)     │      1,677 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 166,957 (652.18 KB)

 Trainable params: 166,957 (652.18 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - loss: 2.5645 - sparse_categorical_accuracy: 0.1154 - val_loss: 2.5656 - val_sparse_categorical_accuracy: 0.0000e+00
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - loss: 2.5581 - sparse_categorical_accuracy: 0.3462 - val_loss: 2.5646 - val_sparse_categorical_accuracy: 0.2000
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step - loss: 2.5514 - sparse_categorical_accuracy: 0.3846 - val_loss: 2.5636 - val_sparse_categorical_accuracy: 0.2000
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step - loss: 2.5441 - sparse_categorical_accuracy: 0.3846 - val_loss: 2.5628 - val_sparse_categorical_accuracy: 0.2000
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step - loss: 2.5359 - sparse_categorical_accuracy: 0.3846 - val_loss: 2.5619 - val_sparse_categorical_accuracy: 0.2000
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step - loss: 2.5264 - sparse_categorical_accuracy: 0.3462 - val_loss: 2.5612 - val_sparse_categorical_accuracy: 0.2000
Epoch 7/1

In [24]:
encoder_model = Model(encoder_tokens, encoder_states, name="encoder_inference")

decoder_token_input = layers.Input(shape=(None,), dtype="int32", name="decoder_token_input")
state_h_input = layers.Input(shape=(latent_dim,), name="state_h_input")
state_c_input = layers.Input(shape=(latent_dim,), name="state_c_input")
decoder_embedding_input = token_embedding(decoder_token_input)
decoder_output, next_state_h, next_state_c = decoder_lstm(
    decoder_embedding_input,
    initial_state=[state_h_input, state_c_input],
)
step_probabilities = token_classifier(decoder_output)
decoder_model = Model(
    [decoder_token_input, state_h_input, state_c_input],
    [step_probabilities, next_state_h, next_state_c],
    name="decoder_inference",
)

def reverse_with_model(text):
    """Generate a reversed character sequence with the trained decoder."""
    normalized_text = text.lower()
    unknown_characters = sorted(set(normalized_text) - set(char_to_id))
    if unknown_characters:
        raise ValueError(f"Input contains characters not seen in training: {unknown_characters}")
    if len(normalized_text) > max_input_length:
        raise ValueError(f"Input exceeds the trained maximum length of {max_input_length} characters.")

    encoded = pad_sequences(
        [[char_to_id[character] for character in normalized_text]],
        maxlen=max_input_length,
        padding="post",
        dtype="int32",
    )
    state_h, state_c = encoder_model.predict(encoded, verbose=0)
    current_token = np.array([[START_ID]], dtype="int32")
    generated_characters = []

    for _ in range(len(normalized_text) + 1):
        probabilities, state_h, state_c = decoder_model.predict(
            [current_token, state_h, state_c],
            verbose=0,
        )
        predicted_id = int(np.argmax(probabilities[0, -1, :]))
        if predicted_id == END_ID:
            break
        if predicted_id == PAD_ID or predicted_id == START_ID:
            break
        generated_characters.append(id_to_char[predicted_id])
        current_token = np.array([[predicted_id]], dtype="int32")

    return "".join(generated_characters)


In [26]:
for example in character_sequences[:5]:
    prediction = reverse_with_model(example)
    print(f"Input:     {example}")
    print(f"Expected:  {example[::-1]}")
    print(f"Predicted: {prediction}\n")

Input:     12345
Expected:  54321
Predicted: 44222

Input:     5678
Expected:  8765
Predicted: 422

Input:     2349234
Expected:  4329432
Predicted: 44222

Input:     234509
Expected:  905432
Predicted: 44222

Input:     8756
Expected:  6578
Predicted: 422

